<a href="https://colab.research.google.com/github/shresthshukla18/Industrial-Defect-Detection-System/blob/main/POC_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CELL1

# ==============================
# 🔹 COLAB SETUP (ROBUST)
# ==============================

import os
import shutil
import torch

# ===== 1. VERIFY COLAB =====
try:
    import google.colab
    print("✅ Running in Google Colab")
except:
    raise Exception(" Not running in Colab. Open notebook in Google Colab.")

# ===== 2. GPU CHECK =====
print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ WARNING: No GPU. Go to Runtime > Change runtime > GPU")

# ===== 3. INSTALL DEPENDENCIES =====
!pip install -q anomalib timm torchmetrics opencv-python kagglehub

print("✅ Dependencies installed")

# ===== 4. CLEAN REMOUNT (FIXES YOUR ERROR) =====
from google.colab import drive

try:
    drive.mount('/content/drive', force_remount=True)
    print(" Drive mounted successfully")
except Exception as e:
    print(" Drive mount failed. Retrying...")
    drive.mount('/content/drive', force_remount=True)

# ===== 5. DEFINE PATHS =====
PROJECT_PATH = "/content/drive/MyDrive/POC5_Project"
DATASET_BASE = os.path.join(PROJECT_PATH, "dataset/mvtec-ad")

print(" Project path:", PROJECT_PATH)

# ===== 6. CREATE FOLDERS =====
folders = [
    "dataset/mvtec-ad",
    "models/autoencoder",
    "models/padim",
    "models/patchcore",
    "outputs/images",
    "outputs/heatmaps",
    "outputs/localization",
    "outputs/overlays",
    "results"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print(" Folder structure ready")

# ===== 7. DOWNLOAD DATASET (SAFE) =====
if os.path.exists(DATASET_BASE) and len(os.listdir(DATASET_BASE)) > 0:
    print(" Dataset already exists")

else:
    print("⬇ Downloading dataset...")

    import kagglehub
    path = kagglehub.dataset_download("ipythonx/mvtec-ad")

    print("Downloaded to:", path)

    shutil.copytree(path, DATASET_BASE, dirs_exist_ok=True)
    print(" Dataset copied to Drive")

print(" Dataset path:", DATASET_BASE)

# ===== 8. VERIFY DATASET =====
all_categories = [
    c for c in os.listdir(DATASET_BASE)
    if os.path.isdir(os.path.join(DATASET_BASE, c))
]

if len(all_categories) == 0:
    raise Exception(" Dataset is empty or corrupted")

print("\nTotal categories:", len(all_categories))
print(all_categories)

# ===== 9. SELECT CATEGORY =====
CATEGORY = all_categories[0]
print("\nUsing CATEGORY:", CATEGORY)

# ===== 10. VERIFY TRAIN DATA =====
train_path = os.path.join(DATASET_BASE, CATEGORY, "train", "good")

if not os.path.exists(train_path):
    raise Exception(" Train folder missing")

train_images = os.listdir(train_path)
print(f" Found {len(train_images)} training images")

In [ ]:
#CELL2

import os
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2

os.environ["TQDM_DISABLE"] = "1"

# ===== ANOMALIB =====
from anomalib.models import Padim, Patchcore
from anomalib.engine import Engine
from anomalib.data import MVTecAD

# ===== DEVICE =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== CONTROL SWITCH =====
RETRAIN = False   #  Set True only when you WANT retraining


# ==============================
# AUTOENCODER
# ==============================
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


# ==============================
# DATASET
# ==============================
class MVTecTrainDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir

        files = [f for f in os.listdir(root_dir) if f.endswith(('.png', '.jpg'))]

        self.images = []
        for f in files:
            path = os.path.join(root_dir, f)
            if cv2.imread(path) is not None:
                self.images.append(f)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.images[idx])
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img)


# ==============================
# GET CATEGORIES
# ==============================
all_categories = [
    c for c in os.listdir(DATASET_BASE)
    if os.path.isdir(os.path.join(DATASET_BASE, c))
]

print("Total categories:", len(all_categories))


# ==============================
# MAIN LOOP
# ==============================
for CATEGORY in all_categories:

    print("\n==============================")
    print("Processing:", CATEGORY)
    print("==============================")

    train_path = f"{DATASET_BASE}/{CATEGORY}/train/good"

    if not os.path.exists(train_path):
        print("Skipping:", CATEGORY)
        continue

    # =========================
    # AUTOENCODER
    # =========================
    ae_path = f"{PROJECT_PATH}/models/autoencoder/{CATEGORY}.pth"

    if os.path.exists(ae_path) and not RETRAIN:
        print(f"{CATEGORY} AE exists → skipping")
    else:
        print(f"{CATEGORY} AE training...")

        dataset = MVTecTrainDataset(train_path)
        loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)

        ae_model = Autoencoder().to(device)
        optimizer = optim.Adam(ae_model.parameters(), lr=0.001)

        for epoch in range(30):
            total_loss = 0

            for imgs in loader:
                imgs = imgs.to(device)

                outputs = ae_model(imgs)
                loss = F.mse_loss(outputs, imgs)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            print(f"{CATEGORY} AE Epoch {epoch+1}: {total_loss/len(loader):.6f}")

        os.makedirs(os.path.dirname(ae_path), exist_ok=True)
        torch.save(ae_model.state_dict(), ae_path)
        print(f"{CATEGORY} AE saved")


    # =========================
    # PADIM
    # =========================
    padim_path = f"{PROJECT_PATH}/models/padim/{CATEGORY}.ckpt"

    if os.path.exists(padim_path) and not RETRAIN:
        print(f"{CATEGORY} PaDiM exists → skipping")
    else:
        print(f"{CATEGORY} PaDiM training...")

        datamodule = MVTecAD(
            root=DATASET_BASE,
            category=CATEGORY,
            train_batch_size=32,
            eval_batch_size=32,
            num_workers=2
        )
        datamodule.setup()

        padim_model = Padim(
            backbone="resnet18",
            layers=["layer1", "layer2", "layer3"]
        )

        padim_engine = Engine(
            max_epochs=1,
            accelerator="gpu" if torch.cuda.is_available() else "cpu",
            devices=1
        )

        padim_engine.fit(model=padim_model, datamodule=datamodule)

        padim_ckpt = padim_engine.trainer.checkpoint_callback.best_model_path

        if padim_ckpt:
            os.makedirs(os.path.dirname(padim_path), exist_ok=True)
            shutil.copy(padim_ckpt, padim_path)
            print(f"{CATEGORY} PaDiM saved")


    # =========================
    # PATCHCORE
    # =========================
    patch_path = f"{PROJECT_PATH}/models/patchcore/{CATEGORY}.ckpt"

    if os.path.exists(patch_path) and not RETRAIN:
        print(f"{CATEGORY} PatchCore exists → skipping")
    else:
        print(f"{CATEGORY} PatchCore training...")

        try:
            datamodule = MVTecAD(
                root=DATASET_BASE,
                category=CATEGORY,
                train_batch_size=32,
                eval_batch_size=32,
                num_workers=2
            )
            datamodule.setup()

            patch_model = Patchcore(
                backbone="resnet18",
                layers=["layer2", "layer3"],
                coreset_sampling_ratio=0.01
            )

            patch_engine = Engine(
                max_epochs=1,
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                devices=1,
                enable_progress_bar=False
            )

            patch_engine.fit(model=patch_model, datamodule=datamodule)

            patch_ckpt = patch_engine.trainer.checkpoint_callback.best_model_path

            if patch_ckpt:
                os.makedirs(os.path.dirname(patch_path), exist_ok=True)
                shutil.copy(patch_ckpt, patch_path)
                print(f"{CATEGORY} PatchCore saved")

        except Exception as e:
            print(f"{CATEGORY} PatchCore FAILED:", str(e))

In [ ]:
#CELL3
import os
import random
import torch

# ===== DEVICE =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===== RANDOMIZE CATEGORY =====
# Find all category folders dynamically
all_categories = sorted([d for d in os.listdir(DATASET_BASE) if os.path.isdir(os.path.join(DATASET_BASE, d))])

if len(all_categories) == 0:
    raise Exception("No categories found in your dataset folder!")

# Pick a completely random category!
CATEGORY = random.choice(all_categories)
print("\n==============================")
print("🎲 Randomly Selected CATEGORY:", CATEGORY.upper())
print("==============================")

test_path = os.path.join(DATASET_BASE, CATEGORY, "test")
if not os.path.exists(test_path):
    raise Exception(f"Test path not found: {test_path}")

# ===== GET DEFECT TYPES =====
# Find all folders that are not "good"
defect_types = [d for d in os.listdir(test_path) if d != "good"]
if len(defect_types) == 0:
    raise Exception(f"No defect types found for {CATEGORY}")

print("Available defects:", defect_types)

# ===== RANDOMIZE DEFECT & IMAGE =====
selected_defect = random.choice(defect_types)
defect_path = os.path.join(test_path, selected_defect)
test_images = sorted([f for f in os.listdir(defect_path) if f.endswith(('.png', '.jpg'))])

if len(test_images) == 0:
    raise Exception("No test images found")

img_name = random.choice(test_images)
test_img_path = os.path.join(defect_path, img_name)

print("🎲 Selected defect:", selected_defect)
print("🖼️ Selected image:", img_name)

In [ ]:
!rm -rf ~/.cache/huggingface

In [ ]:
# ==============================
# FINAL INFERENCE CELL (ENGINE BASED)
# ==============================
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from anomalib.models import Padim, Patchcore
from anomalib.engine import Engine
from anomalib.data import MVTecAD

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ==============================
# DEVICE
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==============================
# LOAD IMAGE (FOR AE ONLY)
# ==============================
if not os.path.exists(test_img_path):
    raise Exception("Test image not found")

img = cv2.imread(test_img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

input_tensor = transform(img).unsqueeze(0).to(device)
input_np = input_tensor.cpu().squeeze().permute(1, 2, 0).numpy()

# ==============================
# AUTOENCODER
# ==============================
ae_path = f"{PROJECT_PATH}/models/autoencoder/{CATEGORY}.pth"

ae_model = Autoencoder().to(device)
ae_model.load_state_dict(torch.load(ae_path, map_location=device))
ae_model.eval()

with torch.no_grad():
    ae_recon = ae_model(input_tensor)

ae_recon_np = ae_recon.cpu().squeeze().permute(1, 2, 0).numpy()
ae_error = np.abs(input_np - ae_recon_np).mean(axis=2)

# AE localization
threshold = np.percentile(ae_error, 95)
mask = (ae_error > threshold).astype(np.uint8)

kernel = np.ones((5, 5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

mask = (mask * 255).astype(np.uint8)

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

ae_box = (input_np * 255).astype(np.uint8).copy()

for cnt in contours:
    if cv2.contourArea(cnt) > 120:
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(ae_box, (x, y), (x + w, y + h), (0, 255, 0), 2)

# ==============================
# PADIM (ENGINE)
# ==============================
padim_ckpt = f"{PROJECT_PATH}/models/padim/{CATEGORY}.ckpt"

datamodule = MVTecAD(
    root=DATASET_BASE,
    category=CATEGORY,
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=0
)

datamodule.setup()

padim_model = Padim(
    backbone="resnet18",
    layers=["layer1", "layer2", "layer3"]
)

padim_engine = Engine(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1
)

padim_preds = padim_engine.predict(
    model=padim_model,
    datamodule=datamodule,
    ckpt_path=padim_ckpt
)

padim_sample = padim_preds[0]
padim_img = padim_sample["image"][0]
padim_map = padim_sample["anomaly_map"][0]

padim_img = padim_img.cpu().numpy().transpose(1, 2, 0)
padim_map = padim_map.cpu().numpy()

# ==============================
# PATCHCORE (ENGINE)
# ==============================
patch_ckpt = f"{PROJECT_PATH}/models/patchcore/{CATEGORY}.ckpt"

patch_model = Patchcore(
    backbone="resnet18",
    layers=["layer2", "layer3"],
    coreset_sampling_ratio=0.01
)

patch_engine = Engine(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    enable_progress_bar=False
)

patch_preds = patch_engine.predict(
    model=patch_model,
    datamodule=datamodule,
    ckpt_path=patch_ckpt
)

patch_sample = patch_preds[0]
patch_img = patch_sample["image"][0]
patch_map = patch_sample["anomaly_map"][0]

patch_img = patch_img.cpu().numpy().transpose(1, 2, 0)
patch_map = patch_map.cpu().numpy()

# ==============================
# FINAL VISUALIZATION (RAW + OVERLAY)
# ==============================
def normalize_map(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def overlay_heatmap(image, heatmap):
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap = (heatmap * 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted((image * 255).astype(np.uint8), 0.75, heatmap, 0.25, 0)
    return overlay

# Normalize maps
ae_norm = normalize_map(ae_error)
padim_norm = normalize_map(padim_map)
patch_norm = normalize_map(patch_map)

# Overlays
padim_overlay = overlay_heatmap(input_np, padim_norm)
patch_overlay = overlay_heatmap(input_np, patch_norm)

# ==============================
# PLOTTING (3 ROWS)
# ==============================
plt.figure(figsize=(18, 14))

# Row 1 -> AE
plt.subplot(3, 3, 1)
plt.title("Original")
plt.imshow(input_np)
plt.axis("off")

plt.subplot(3, 3, 2)
plt.title("Reconstruction")
plt.imshow(ae_recon_np)
plt.axis("off")

plt.subplot(3, 3, 3)
plt.title("AE Raw Heatmap")
plt.imshow(ae_norm, cmap="jet")
plt.axis("off")

# Row 2 -> RAW MAPS
plt.subplot(3, 3, 4)
plt.title("AE Localization")
plt.imshow(ae_box)
plt.axis("off")

plt.subplot(3, 3, 5)
plt.title("PaDiM Raw Heatmap")
plt.imshow(padim_norm, cmap="jet")
plt.axis("off")

plt.subplot(3, 3, 6)
plt.title("PatchCore Raw Heatmap")
plt.imshow(patch_norm, cmap="jet")
plt.axis("off")

# Row 3 -> OVERLAYS
plt.subplot(3, 3, 7)
plt.title("AE (Reference)")
plt.imshow(input_np)
plt.axis("off")

plt.subplot(3, 3, 8)
plt.title("PaDiM Overlay")
plt.imshow(padim_overlay)
plt.axis("off")

plt.subplot(3, 3, 9)
plt.title("PatchCore Overlay")
plt.imshow(patch_overlay)
plt.axis("off")

plt.tight_layout()
plt.show()

# ==============================
# SAVE OUTPUTS (FIXED NAMING)
# ==============================
save_base = os.path.join(PROJECT_PATH, "outputs")

base_name = f"{selected_defect}_{os.path.basename(test_img_path).split('.')[0]}"

img_dir = os.path.join(save_base, "images", CATEGORY)
heatmap_dir = os.path.join(save_base, "heatmaps", CATEGORY)
loc_dir = os.path.join(save_base, "localization", CATEGORY)
overlay_dir = os.path.join(save_base, "overlays", CATEGORY)

os.makedirs(img_dir, exist_ok=True)
os.makedirs(heatmap_dir, exist_ok=True)
os.makedirs(loc_dir, exist_ok=True)
os.makedirs(overlay_dir, exist_ok=True)

# ORIGINAL
cv2.imwrite(
    os.path.join(img_dir, f"{base_name}_original.png"),
    cv2.cvtColor((input_np * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
)

# AE HEATMAP
ae_heatmap_color = cv2.applyColorMap(
    (ae_norm * 255).astype(np.uint8),
    cv2.COLORMAP_JET
)

cv2.imwrite(
    os.path.join(heatmap_dir, f"{base_name}_ae.png"),
    ae_heatmap_color
)

# PADIM HEATMAP
cv2.imwrite(
    os.path.join(heatmap_dir, f"{base_name}_padim.png"),
    (padim_norm * 255).astype(np.uint8)
)

# PATCHCORE HEATMAP
cv2.imwrite(
    os.path.join(heatmap_dir, f"{base_name}_patchcore.png"),
    (patch_norm * 255).astype(np.uint8)
)

# AE LOCALIZATION
cv2.imwrite(
    os.path.join(loc_dir, f"{base_name}_ae_box.png"),
    cv2.cvtColor(ae_box, cv2.COLOR_RGB2BGR)
)

# OVERLAYS
cv2.imwrite(
    os.path.join(overlay_dir, f"{base_name}_padim_overlay.png"),
    cv2.cvtColor(padim_overlay, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    os.path.join(overlay_dir, f"{base_name}_patchcore_overlay.png"),
    cv2.cvtColor(patch_overlay, cv2.COLOR_RGB2BGR)
)

print(f"Saved outputs for {CATEGORY} → {base_name}")

In [ ]:
# ==============================
# FINAL EVALUATION (ALL MODELS)
# ==============================
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torchvision import transforms
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from anomalib.models import Padim, Patchcore
from anomalib.data import MVTecAD
from anomalib.engine import Engine

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating category: {CATEGORY}")

# ==============================
# DATAMODULE
# ==============================
datamodule = MVTecAD(
    root=DATASET_BASE,
    category=CATEGORY,
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=2
)
datamodule.setup()

engine = Engine(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1
)

# ==============================
# PADIM
# ==============================
padim_ckpt = f"{PROJECT_PATH}/models/padim/{CATEGORY}.ckpt"
padim_model = Padim.load_from_checkpoint(padim_ckpt)

padim_results = engine.test(
    model=padim_model,
    datamodule=datamodule
)[0]

# ==============================
# PATCHCORE
# ==============================
patch_ckpt = f"{PROJECT_PATH}/models/patchcore/{CATEGORY}.ckpt"

patch_model = Patchcore.load_from_checkpoint(
    patch_ckpt,
    map_location=device,
    weights_only=False
)

patch_results = engine.test(
    model=patch_model,
    datamodule=datamodule
)[0]

# ==============================
# AUTOENCODER
# ==============================
ae_path = f"{PROJECT_PATH}/models/autoencoder/{CATEGORY}.pth"

ae_model = Autoencoder().to(device)
ae_model.load_state_dict(torch.load(ae_path, map_location=device))
ae_model.eval()

test_dir = os.path.join(DATASET_BASE, CATEGORY, "test")

y_true = []
y_scores = []

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

for defect_type in os.listdir(test_dir):
    defect_path = os.path.join(test_dir, defect_type)

    for img_name in os.listdir(defect_path):
        img_path = os.path.join(defect_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        input_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            recon = ae_model(input_tensor)

        recon_np = recon.cpu().squeeze().permute(1, 2, 0).numpy()
        input_np = input_tensor.cpu().squeeze().permute(1, 2, 0).numpy()

        error_map = np.abs(input_np - recon_np).mean(axis=2)
        score = error_map.mean()

        y_scores.append(score)
        y_true.append(0 if defect_type == "good" else 1)

# Metrics
ae_roc = roc_auc_score(y_true, y_scores)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
ae_pr = auc(recall, precision)

# ==============================
# FINAL PRINT
# ==============================
print("\n===== FINAL RESULTS =====")

print("\n--- PaDiM ---")
print(padim_results)

print("\n--- PatchCore ---")
print(patch_results)

print("\n--- Autoencoder ---")
print({
    "image_AUROC": ae_roc,
    "PR_AUC": ae_pr
})

# ==============================
# SAVE RESULTS (CSV)
# ==============================
results_dir = os.path.join(PROJECT_PATH, "results")
os.makedirs(results_dir, exist_ok=True)

data = [
    {
        "Model": "Autoencoder",
        "Image_AUROC": float(ae_roc),
        "Pixel_AUROC": None,
        "Image_F1": None,
        "Pixel_F1": None,
        "PR_AUC": float(ae_pr)
    },
    {
        "Model": "PaDiM",
        "Image_AUROC": padim_results["image_AUROC"],
        "Pixel_AUROC": padim_results["pixel_AUROC"],
        "Image_F1": padim_results["image_F1Score"],
        "Pixel_F1": padim_results["pixel_F1Score"],
        "PR_AUC": None
    },
    {
        "Model": "PatchCore",
        "Image_AUROC": patch_results["image_AUROC"],
        "Pixel_AUROC": patch_results["pixel_AUROC"],
        "Image_F1": patch_results["image_F1Score"],
        "Pixel_F1": patch_results["pixel_F1Score"],
        "PR_AUC": None
    }
]

df = pd.DataFrame(data)

csv_path = os.path.join(results_dir, f"{CATEGORY}_evaluation.csv")
df.to_csv(csv_path, index=False)

print(f"✅ Results saved to: {csv_path}")

# ==============================
# DISPLAY TABLE
# ==============================
print("\n===== RESULT TABLE =====")
display(df)

# ==============================
# GRAPH (Image AUROC)
# ==============================
models = df["Model"]
image_auroc = df["Image_AUROC"]

plt.figure()
plt.bar(models, image_auroc)

for i, v in enumerate(image_auroc):
    if v is not None:
        plt.text(i, v + 0.01, f"{v:.3f}", ha='center')

plt.title("Model Comparison (Image AUROC)")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1.05)

graph_path = os.path.join(results_dir, f"{CATEGORY}_auroc_graph.png")
plt.savefig(graph_path, bbox_inches='tight')
plt.show()

print(f"✅ Graph saved to: {graph_path}")

# ==============================
# ANOMALY SCORE DISTRIBUTION
# ==============================

# ---- PaDiM ----
padim_scores, padim_labels = [], []

padim_preds_for_plot = engine.predict(
    model=padim_model,
    datamodule=datamodule,
    ckpt_path=padim_ckpt
)

for p in padim_preds_for_plot:
    score = float(p["pred_score"][0].cpu().numpy().item())
    img_path = p.image_path[0]
    label = 0 if "good" in img_path else 1

    padim_scores.append(score)
    padim_labels.append(label)

good_scores_padim = [s for s, l in zip(padim_scores, padim_labels) if l == 0]
defect_scores_padim = [s for s, l in zip(padim_scores, padim_labels) if l == 1]

# ---- PatchCore ----
patch_scores, patch_labels = [], []

patch_preds_for_plot = engine.predict(
    model=patch_model,
    datamodule=datamodule,
    ckpt_path=patch_ckpt
)

for p in patch_preds_for_plot:
    score = float(p["pred_score"][0].cpu().numpy().item())
    img_path = p.image_path[0]
    label = 0 if "good" in img_path else 1

    patch_scores.append(score)
    patch_labels.append(label)

good_scores_patch = [s for s, l in zip(patch_scores, patch_labels) if l == 0]
defect_scores_patch = [s for s, l in zip(patch_scores, patch_labels) if l == 1]

# ---- Autoencoder ----
good_scores_ae = [s for s, l in zip(y_scores, y_true) if l == 0]
defect_scores_ae = [s for s, l in zip(y_scores, y_true) if l == 1]

# ==============================
# PLOTTING FUNCTION
# ==============================
def plot_and_save_scores(good_scores, defect_scores, model_name):
    plt.figure(figsize=(8, 6))

    plt.hist(good_scores, bins=30, alpha=0.6, label="Good")
    plt.hist(defect_scores, bins=30, alpha=0.6, label="Defect")

    plt.title(f"Anomaly Score Distribution ({model_name})")
    plt.xlabel("Anomaly Score")
    plt.ylabel("Frequency")
    plt.legend()

    save_path = os.path.join(
        results_dir,
        f"{CATEGORY}_{model_name.lower()}_score_distribution.png"
    )

    plt.savefig(save_path, bbox_inches='tight')
    plt.show()

    print(f"✅ {model_name} Score plot saved to: {save_path}")
    plt.close()

# Run plots
plot_and_save_scores(good_scores_padim, defect_scores_padim, "PaDiM")
plot_and_save_scores(good_scores_patch, defect_scores_patch, "PatchCore")
plot_and_save_scores(good_scores_ae, defect_scores_ae, "Autoencoder")

In [ ]:
# ==========================================
# 🔹 SETUP STREAMLIT & CLOUDFLARE
# ==========================================
!pip install -q streamlit
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print("✅ Streamlit and Cloudflared installed successfully.")

In [ ]:
%%writefile app.py
import streamlit as st
import os
import pandas as pd
from PIL import Image
import time

# ==========================================
# 🔹 CONFIGURATION
# ==========================================
st.set_page_config(page_title="PoC-5 Dashboard", layout="wide")
st.title("🏭 Defect Surveillance Dashboard")

PROJECT_PATH = "/content/drive/MyDrive/POC5_Project"
OUTPUTS_PATH = os.path.join(PROJECT_PATH, "outputs")
RESULTS_PATH = os.path.join(PROJECT_PATH, "results")

# ==========================================
# 🔹 DATA GATHERING & PARSING
# ==========================================
if not os.path.exists(OUTPUTS_PATH) or not os.path.exists(os.path.join(OUTPUTS_PATH, "images")):
    st.error("No processed outputs found. Please run your pipeline first.")
    st.stop()

all_runs = []
image_base_dir = os.path.join(OUTPUTS_PATH, "images")
for category in os.listdir(image_base_dir):
    cat_dir = os.path.join(image_base_dir, category)
    if not os.path.isdir(cat_dir): continue

    for file in os.listdir(cat_dir):
        if file.endswith("_original.png"):
            file_path = os.path.join(cat_dir, file)
            mtime = os.path.getmtime(file_path)

            base_name = file.replace("_original.png", "")
            name_parts = base_name.split("_")

            if len(name_parts) > 1 and name_parts[-1].isdigit():
                defect_type = " ".join(name_parts[:-1]).title()
            elif len(name_parts) == 1 and name_parts[0].isdigit():
                defect_type = "Legacy Run (Unrecorded Defect)"
            else:
                defect_type = base_name.replace("_", " ").title()

            all_runs.append({
                "category": category.capitalize(),
                "category_raw": category,
                "defect": defect_type,
                "base_name": base_name,
                "mtime": mtime,
                "paths": {
                    "orig": file_path,
                    "box": os.path.join(OUTPUTS_PATH, "localization", category, f"{base_name}_ae_box.png"),
                    "padim": os.path.join(OUTPUTS_PATH, "overlays", category, f"{base_name}_padim_overlay.png"),
                    "patch": os.path.join(OUTPUTS_PATH, "overlays", category, f"{base_name}_patchcore_overlay.png")
                }
            })

if not all_runs:
    st.info("No images found in outputs.")
    st.stop()

all_runs.sort(key=lambda x: x["mtime"], reverse=True)

current_run = all_runs[0]
history_runs = all_runs[1:]

# ==========================================
# 🔹 UI LAYOUT (TABS)
# ==========================================
tab1, tab2 = st.tabs(["🚀 Current Run", "📜 History Panel"])

# --- TAB 1: CURRENT RUN ---
with tab1:
    st.header(f"Target Category: {current_run['category']}")
    st.subheader(f"Detected Condition: {current_run['defect']}")

    cols = st.columns(4)
    paths = current_run["paths"]
    if os.path.exists(paths["orig"]): cols[0].image(Image.open(paths["orig"]), caption="Original Image", use_container_width=True)
    if os.path.exists(paths["box"]): cols[1].image(Image.open(paths["box"]), caption="Autoencoder", use_container_width=True)
    if os.path.exists(paths["padim"]): cols[2].image(Image.open(paths["padim"]), caption="PaDiM Overlay", use_container_width=True)
    if os.path.exists(paths["patch"]): cols[3].image(Image.open(paths["patch"]), caption="PatchCore Overlay", use_container_width=True)

    st.divider()

    st.markdown(f"### 📊 Analytics & Performance ({current_run['category']})")
    metrics_file = os.path.join(RESULTS_PATH, f"{current_run['category_raw']}_evaluation.csv")

    if os.path.exists(metrics_file):
        st.dataframe(pd.read_csv(metrics_file), use_container_width=True, hide_index=True)
        col_c1, col_c2 = st.columns([1, 2])

        auroc_img = os.path.join(RESULTS_PATH, f"{current_run['category_raw']}_auroc_graph.png")
        if os.path.exists(auroc_img):
            col_c1.image(Image.open(auroc_img), use_container_width=True)

        dist_cols = col_c2.columns(3)
        for col, model in zip(dist_cols, ["autoencoder", "padim", "patchcore"]):
            dist_img = os.path.join(RESULTS_PATH, f"{current_run['category_raw']}_{model}_score_distribution.png")
            if os.path.exists(dist_img):
                col.image(Image.open(dist_img), caption=model.capitalize(), use_container_width=True)
    else:
        st.warning("Analytics not generated yet. Run Cell 5 to populate this section.")

# --- TAB 2: HISTORY PANEL ---
with tab2:
    st.header("Previous Runs")
    if not history_runs:
        st.info("No previous runs found. Run your pipeline again to start building history.")
    else:
        for run in history_runs:
            run_time = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(run['mtime']))
            with st.expander(f"🕰️ {run['category']} - {run['defect']} (Processed: {run_time})"):

                # 1. Show the images
                h_cols = st.columns(4)
                h_paths = run["paths"]
                if os.path.exists(h_paths["orig"]): h_cols[0].image(Image.open(h_paths["orig"]), caption="Original Image", use_container_width=True)
                if os.path.exists(h_paths["box"]): h_cols[1].image(Image.open(h_paths["box"]), caption="Autoencoder", use_container_width=True)
                if os.path.exists(h_paths["padim"]): h_cols[2].image(Image.open(h_paths["padim"]), caption="PaDiM Overlay", use_container_width=True)
                if os.path.exists(h_paths["patch"]): h_cols[3].image(Image.open(h_paths["patch"]), caption="PatchCore Overlay", use_container_width=True)

                st.divider()

                # 2. Show the analytics for that run's category
                st.markdown(f"#### 📊 Analytics ({run['category']})")
                hist_metrics_file = os.path.join(RESULTS_PATH, f"{run['category_raw']}_evaluation.csv")

                if os.path.exists(hist_metrics_file):
                    st.dataframe(pd.read_csv(hist_metrics_file), use_container_width=True, hide_index=True)
                    col_h1, col_h2 = st.columns([1, 2])

                    hist_auroc_img = os.path.join(RESULTS_PATH, f"{run['category_raw']}_auroc_graph.png")
                    if os.path.exists(hist_auroc_img):
                        col_h1.image(Image.open(hist_auroc_img), use_container_width=True)

                    hist_dist_cols = col_h2.columns(3)
                    for col, model in zip(hist_dist_cols, ["autoencoder", "padim", "patchcore"]):
                        hist_dist_img = os.path.join(RESULTS_PATH, f"{run['category_raw']}_{model}_score_distribution.png")
                        if os.path.exists(hist_dist_img):
                            col.image(Image.open(hist_dist_img), caption=model.capitalize(), use_container_width=True)
                else:
                    st.info("Analytics not found for this historical run.")

In [ ]:
import subprocess
import time
import re
import socket
import os

# 1. Kill old processes to prevent port conflicts
!pkill -f streamlit
!pkill -f cloudflared
time.sleep(2)

# 2. Verify binary name based on your setup step
binary_name = "./cloudflared-linux-amd64"
if not os.path.exists(binary_name):
    if os.path.exists("./cloudflared"):
        binary_name = "./cloudflared"
    else:
        print("❌ Error: Cloudflare binary not found. Please re-run your setup cell.")

# 3. Port checker function
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

# 4. Start Streamlit
print("🚀 Starting Streamlit in the background...")
with open("streamlit_log.txt", "w") as f:
    subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
        stdout=f, stderr=f
    )

# 5. Wait for Streamlit to open port 8501
print("⏳ Waiting for Streamlit to boot (Port 8501)...")
timeout = 30
start_time = time.time()
while not is_port_open(8501):
    if time.time() - start_time > timeout:
        print("❌ Streamlit failed to start within 30 seconds. Check streamlit_log.txt")
        break
    time.sleep(1)
else:
    print("✅ Streamlit is LIVE! Starting tunnel...")

    # 6. Start Cloudflare Tunnel
    process = subprocess.Popen(
        [binary_name, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    url = None
    # Read output to find the URL
    for _ in range(100):
        line = process.stdout.readline()
        if not line:
            break

        if "trycloudflare.com" in line:
            match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
            if match:
                url = match.group(0)
                print(f"\n🔗 YOUR DASHBOARD URL: {url}\n")

        if "Connected" in line or "Connection established" in line:
            print("✅ Tunnel Connected Successfully! You can click the link above.")
            break

    if not url:
        print("❌ Failed to parse Tunnel URL. Try running this cell again.")